# Training and ablations on Colab

One-time setup:
1. Runtime -> Change runtime type -> **T4 GPU**.
2. If the repo is private: make a GitHub token with read access to `banyasp/neuro-neuro`, add it in
   Colab Secrets (key icon, left sidebar) as `GH_TOKEN`, and switch on notebook access.

Every session: run cells 1 and 2. Cells 3 and 4 (download + preprocess) run once; the cache stays on Drive.
After a disconnect, rerun cells 1 and 2 and then the same training cell: finished runs are skipped and an
interrupted one continues from its checkpoint.

## 1. Drive and GPU check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
CACHE = "/content/drive/MyDrive/neuro/cache"
RUNS = "/content/drive/MyDrive/neuro/runs"
PATIENTS = ["chb01", "chb03", "chb05", "chb08", "chb10"]

## 2. Code and packages (Colab already has NumPy, PyTorch, SciPy)

In [ ]:
from google.colab import userdata
try:
    token = userdata.get('GH_TOKEN')
    url = f"https://{token}@github.com/banyasp/neuro-neuro.git"
except Exception:
    url = "https://github.com/banyasp/neuro-neuro.git"   # public repo, no token needed
!git clone -b ryan {url} /content/neuro-neuro || git -C /content/neuro-neuro pull
%cd /content/neuro-neuro/ML
!pip -q install mne

## 3. Download CHB-MIT from PhysioNet (once; about 1-2 GB per patient, to the Colab disk)

In [ ]:
for p in PATIENTS:
    !wget -q -r -N -c -np -nH --cut-dirs=3 -R "index.html*" -P /content/raw https://physionet.org/files/chbmit/1.0.0/{p}/
!ls /content/raw

## 4. Preprocess into a spike cache on Drive (once)

In [ ]:
pats = " ".join(PATIENTS)
!python data/preprocess.py {pats} --raw /content/raw --cache {CACHE}

## 5. Sanity check: every test should pass

In [ ]:
!python test_scnn.py

## 6. Time one run before committing to the big phases

In [ ]:
!time python train.py --patient chb01 --model lif --cache {CACHE} --out {RUNS}

## 7. Phase 1: default models on every patient (20 runs)

In [ ]:
for p in PATIENTS:
    for m in ["lif", "alif", "scnn", "cnn"]:
        !python train.py --patient {p} --model {m} --cache {CACHE} --out {RUNS} --resume --skip-done --verify-n 500
!python summarize.py --out {RUNS} --md {RUNS}/phase1.md
!python diversity.py --patient chb01 --cache {CACHE} --out {RUNS}

## 8. Phase 2: ablation screen on 2 patients, 1 seed (54 runs; defaults already done in phase 1 are skipped)

Either run it in one go:

In [ ]:
!python ablate.py --patients chb01 chb03 --seeds 0 --cache {CACHE} --out {RUNS}

...or as two background jobs sharing the GPU (check progress with the cell after):

In [ ]:
!nohup python ablate.py --patients chb01 chb03 --cache {CACHE} --out {RUNS} --groups scnn > {RUNS}/log_scnn.txt 2>&1 &
!nohup python ablate.py --patients chb01 chb03 --cache {CACHE} --out {RUNS} --groups lif alif > {RUNS}/log_ab.txt 2>&1 &

In [ ]:
!tail -n 5 {RUNS}/log_scnn.txt {RUNS}/log_ab.txt

## 9. Summarize the screen

In [ ]:
!python summarize.py --out {RUNS} --csv {RUNS}/screen.csv --md {RUNS}/screen.md